In [ ]:
import os

import dotenv
dotenv.load_dotenv()
from kbel.disambiguators import Disambiguator
from kbel.core.mention import Mention
from kbel.core.mention import EntityType
from kbel.core.triple import Triple
from kbel.core.kbel import KBEL
from kbel.ingestors import Ingestor
from kbel.ingestors.pdf import PDFIngestor
#import logging
#logging.basicConfig(level=logging.DEBUG)


### Creating a KBEL instance (without disambiguator)

In [ ]:
kbel = KBEL('wikidata-vdb')

#### Getting candidates to a mention:

In [ ]:
results = kbel.candidates_lookup(Mention(term='rock', text='', entity_type=EntityType.ITEM), limit=5)
display (*results)

### Using `naive` disambiguator to link entities from Wikidata

In [ ]:
from kbel.disambiguators.naive import NaiveDisambiguator

kbel = KBEL('wikidata-vdb', 'naive')
kbel.disambiguator = Disambiguator("naive")

results = kbel.link(
    mention=Mention(term='rock', text='', entity_type=EntityType.ITEM))
display (*results)

### Using `similarity` disambiguator to link entities from Wikidata

In [ ]:
#pip install "kbel[similarity]"
from kbel.disambiguators.similarity import SimilarityDisambiguator
kbel.disambiguator = Disambiguator('sim')


In [ ]:
results = kbel.link(
    mention=Mention(term='Rock', text='Rock is a stone', entity_type=EntityType.ITEM),
    limit=100)
display (*results)

##### Linking a property instead of an Item

In [ ]:
results = kbel.link(
    mention=Mention(term='instance of', text='Rock is a stone', entity_type=EntityType.PROPERTY), limit=1
)

display(*results)

### Using `LLM` disambiguator to link entities from Wikidata

Instantiating LLM Disambiguator with OpenAI's models

In [ ]:
# pip install "kbel[llm]"
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model='gpt-5.2', api_key=os.environ['LLM_API_KEY'])

from kbel.disambiguators.llm import LLM_Disambiguator
kbel.disambiguator = Disambiguator('llm', model= model)

In [ ]:
results = kbel.link(
    mention=Mention(term='Rock', text='A rock can be used in construction to mimic the appearance and durability of natural stone.', 
                    entity_type=EntityType.ITEM), limit=2)

display (*results)

##### Adding context to improve disambiguation:

In [ ]:
results = kbel.link(
    Mention(
        term="Python",
        text="Python is used for coding.",
        entity_type=EntityType.ITEM,
        context="""
        Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.
        """
    )
)

display(*results)

### Extracting triples from a text (`TripleExtractor`)

#### `openie` extractor (rule-based, requires Java; the CoreNLP server is started automatically)

In [ ]:
from kbel.extractors import TripleExtractor
from kbel.extractors.openie import OpenIETripleExtractor

kbel.extractor = TripleExtractor('openie')

triples = kbel.extract("Barack Obama was born in Hawaii.")
display(*triples)

##### Rendering the triples as a graph (requires GraphViz)

In [ ]:
kbel.extract("Barack Obama was born in Hawaii.", graph=True, graph_path='graph.png')

#### `llm` extractor

In [ ]:
kbel.extractor = TripleExtractor('llm', model=model)

triples = kbel.extract("Python was created by Guido van Rossum.")
display(*triples)

#### Extracting and linking in one step

In [ ]:
kbel.extractor = TripleExtractor('openie')

results = kbel.extract_and_link("Barack Obama was born in Hawaii.", limit=1)

for triple, (subject, predicate, obj) in results:
    print(triple)
    display(*subject, *predicate, *obj)

### Pipeline - Entity Linking

In [ ]:
texts = [
    "Barack Obama was born in Hawaii.",
    "Python was created by Guido van Rossum.",
    "Marie Curie discovered polonium.",
]

kbel = KBEL('wikidata', 'naive', 'openie')

for text in texts:
    triples = kbel.extract(text)
    mentions = kbel.extractor.build_mentions(triples, text)

    print(f'\n{text}')
    for triple, triple_mentions in zip(triples, mentions):
        print(f'  {triple}')
        for mention in triple_mentions:
            results = kbel.link(mention, limit=1)
            print(f'    {mention.term!r} ->', *results)

### Changing the knowledge source, e.g., DBpedia

In [ ]:
kbel.knowledge_base = 'dbpedia'
results = kbel.link(
    Mention(
        term="Python",
        text="Python is used for coding.",
        entity_type=EntityType.ITEM,
        context="""
        Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.
        """
    )
)

display(*results)